In [ ]:
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import os

In [ ]:
def load_image(image_path):
    img = Image.open(image_path).convert("RGB")
    return img

def add_gaussian_noise(image, noise_level=0.1):
    img_array = np.array(image).astype(np.float32) / 255.0
    noise = np.random.normal(0, noise_level, img_array.shape)
    noisy_img = img_array + noise
    noisy_img = np.clip(noisy_img, 0, 1)
    noisy_img_pil = Image.fromarray((noisy_img * 255).astype(np.uint8))
    return noisy_img_pil

def denoise_with_stable_diffusion(pipeline, image, prompt, strength=0.75, guidance_scale=7.5, num_inference_steps=50):
    width, height = image.size
    new_width = (width // 8) * 8
    new_height = (height // 8) * 8
    image = image.resize((new_width, new_height), Image.LANCZOS)
    result = pipeline(
        prompt=prompt,
        image=image,
        strength=strength, 
        guidance_scale=guidance_scale
        num_inference_steps=num_inference_steps
    ).images[0]
    
    return result


In [ ]:
image_path = "images/dog.jpeg"
noise_level = 0.15 
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("output", exist_ok=True)

original_image = load_image(image_path)
original_image.save("output/01_original.png")

noisy_image = add_gaussian_noise(original_image, noise_level=noise_level)
noisy_image.save("output/02_noisy.png")

pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
)
pipeline = pipeline.to(device)

if hasattr(pipeline, "enable_attention_slicing"):
    pipeline.enable_attention_slicing()

prompt_bulldog = "a bulldog wearing a black pirate hat"
denoised_bulldog = denoise_with_stable_diffusion(
    pipeline, 
    noisy_image, 
    prompt_bulldog,
    strength=0.5, 
    guidance_scale=7.5,
    num_inference_steps=30
)
denoised_bulldog.save("output/03_denoised_bulldog.png")

prompt_mario = "super mario"
denoised_mario = denoise_with_stable_diffusion(
    pipeline, 
    noisy_image, 
    prompt_mario,
    strength=0.5,
    guidance_scale=7.5,
    num_inference_steps=50
)
denoised_mario.save("output/04_denoised_mario.png")

1. Loading original image...
2. Adding Gaussian noise...
3. Loading Stable Diffusion pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

4. Denoising with bulldog prompt...


  0%|          | 0/15 [00:00<?, ?it/s]

5. Denoising with mario prompt...


  0%|          | 0/25 [00:00<?, ?it/s]